# TinyAgCNN — Training Notebook
**Input:** 96×96 Grayscale (1 ch)  
**Teacher:** MobileNetV2 (`agroStake-0.1-p2-finetuned.pth`) — PyTorch  
**Student:** TinyAgCNN — TensorFlow  
**Dataset:** `augmented-dataset-96/` — same split as GrapeStake_1_

Flow:
1. Scan dataset → stratified split (identical to GrapeStake)
2. Extract teacher logits once → save as `.npy` (cross-framework bridge)
3. Build `tf.data` pipelines — grayscale student / RGB teacher (already extracted)
4. **Baseline:** normal CE training
5. **Distillation:** KD from teacher logits
6. Eval + confusion matrix + save

## 1 — Imports & Config

In [2]:
!pip install "tensorflow[and-cuda]"

  Using cached nvidia_cublas_cu12-12.9.2.10-py3-none-win_amd64.whl.metadata (1.8 kB)
  Using cached nvidia_cuda_cupti_cu12-12.9.79-py3-none-win_amd64.whl.metadata (1.8 kB)
  Using cached nvidia_cuda_nvcc_cu12-12.9.86-py3-none-win_amd64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.9.86-py3-none-win_amd64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.9.79-py3-none-win_amd64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.21.1.3-py3-none-win_amd64.whl.metadata (1.9 kB)
  Using cached nvidia_cufft_cu12-11.4.1.4-py3-none-win_amd64.whl.metadata (1.8 kB)
  Using cached nvidia_curand_cu12-10.3.10.19-py3-none-win_amd64.whl.metadata (1.7 kB)
  Using cached nvidia_cusolver_cu12-11.7.5.82-py3-none-win_amd64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparse_cu12-12.5.10.65-py3-none-win_amd64.whl.metadata (1.8 kB)
INFO: pip is looking at multiple versions of tensorflow[and-cuda] to determine which version is compatible with other requirements. This co

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 9.7.0 requires typing_extensions>=4.6; python_version < "3.12", but you have typing-extensions 4.5.0 which is incompatible.
contourpy 1.3.3 requires numpy>=1.25, but you have numpy 1.24.3 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.24.3 which is incompatible.
optree 0.19.0 requires typing-extensions>=4.6.0, but you have typing-extensions 4.5.0 which is incompatible.
scipy 1.16.3 requires numpy<2.6,>=1.25.2, but you have numpy 1.24.3 which is incompatible.
selenium 4.39.0 requires typing_extensions<5.0,>=4.15.0, but you have typing-extensions 4.5.0 which is incompatible.
torch 2.5.1+cu121 requires typing-extensions>=4.8.0, but you have typing-extensions 4.5.0 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1

In [3]:
# ── Standard ──────────────────────────────────────────────────────────────
import os, sys, random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, confusion_matrix, classification_report

# ── TensorFlow ─────────────────────────────────────────────────────────────
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
print(f'TF device: {"GPU" if gpus else "CPU"}')

# ── PyTorch (teacher only) ─────────────────────────────────────────────────
import torch
import torch.nn as nn
import torchvision.models as tv_models
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets as tv_datasets

torch_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {torch_device}')

c:\Users\xared\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\_statistics.py:32: UserWarning: A NumPy version >=1.25.2 and <2.6.0 is required for this version of SciPy (detected version 1.24.3)
  from scipy.stats import gaussian_kde


ModuleNotFoundError: No module named 'numpy.exceptions'

In [3]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATA_ROOT       = '../0_data/augmented-dataset/'
TEACHER_WEIGHTS = '../4_models/agroStake-0.1-p2-finetuned.pth'
LOGITS_CACHE    = './teacher_logits_cache.npy'   # extracted once, reused
IDX_CACHE       = './split_indices.npz'          # save split so both runs use the same

# ── Hyper-params ───────────────────────────────────────────────────────────
IMG_SIZE        = 96
NUM_CLASSES     = 4
BATCH_SIZE      = 32
SEED            = 42

# Baseline (normal CE)
BASELINE_EPOCHS = 30
BASELINE_LR     = 1e-3

# Distillation
KD_EPOCHS       = 30
KD_LR           = 1e-3
KD_ALPHA        = 0.5    # weight for hard CE loss
KD_T            = 4.0    # temperature

CLASS_NAMES = ['Black_rot', 'Esca', 'Leaf_blight', 'healthy']  # update if needed

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2 — Dataset Scan & Stratified Split

In [4]:
# Use torchvision ImageFolder just for the sample list & labels
# (same as GrapeStake — guarantees identical class ordering)
full_train_raw = tv_datasets.ImageFolder(root=os.path.join(DATA_ROOT, 'train'))
CLASS_NAMES    = full_train_raw.classes
print(f'Classes: {CLASS_NAMES}')
print(f'Total train samples: {len(full_train_raw)}')

# ── Stratified split — IDENTICAL to GrapeStake_1_ ─────────────────────────
if os.path.exists(IDX_CACHE):
    cache = np.load(IDX_CACHE)
    train_idx = cache['train_idx'].tolist()
    val_idx   = cache['val_idx'].tolist()
    print('Loaded cached split indices.')
else:
    indices = list(range(len(full_train_raw)))
    train_idx, val_idx = train_test_split(
        indices,
        test_size=0.1,
        stratify=full_train_raw.targets,
        random_state=SEED
    )
    np.savez(IDX_CACHE, train_idx=train_idx, val_idx=val_idx)
    print('Split indices saved.')

# Flat lists of (path, label) for each split
all_samples  = full_train_raw.samples   # list of (path, class_idx)
train_samples = [all_samples[i] for i in train_idx]
val_samples   = [all_samples[i] for i in val_idx]

# Test set
test_dir = os.path.join(DATA_ROOT, 'test')
test_raw  = tv_datasets.ImageFolder(root=test_dir)
test_samples = test_raw.samples

print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}')

Classes: ['Black Rot', 'ESCA', 'Healthy', 'Leaf Blight']
Total train samples: 14444
Split indices saved.
Train: 12999 | Val: 1445 | Test: 3610


## 3 — Teacher Logit Extraction (PyTorch MobileNetV2)

In [5]:
# MobileNetV2 architecture must match how it was finetuned in GrapeStake
# Classifier replaced: features → AdaptiveAvgPool → Flatten → Linear(1280,4)

teacher = tv_models.mobilenet_v2(weights=None)
teacher.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(teacher.last_channel, NUM_CLASSES)
)
teacher.load_state_dict(torch.load(TEACHER_WEIGHTS, map_location=torch_device))
teacher.eval()
teacher.to(torch_device)
print('Teacher loaded.')

C:\Users\xared\AppData\Local\Temp\ipykernel_29348\2790522418.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher.load_state_dict(torch.load(TEACHER_WEIGHTS, map_loca

Teacher loaded.


In [ ]:
# Teacher expects RGB 224×224 (how it was trained in GrapeStake phase-2)
import torchvision.transforms as T

teacher_tf = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

class RGBDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image_pil(img)  # PIL for torchvision transforms
        if self.transform:
            img = self.transform(img)
        return img, label

# PIL bridge
from PIL import Image as Image_pil_module
def Image_pil(arr): return Image_pil_module.fromarray(arr)

# Reload cell with correct name
class RGBDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image_pil_module.fromarray(img)
        if self.transform:
            img = self.transform(img)
        return img, label

: 

In [ ]:
from PIL import Image as Image_pil_module

def extract_logits(samples, teacher, transform, device, batch_size=32, desc='Extracting'):
    """Returns numpy array of shape (N, num_classes) — raw logits."""
    ds = RGBDataset(samples, transform=transform)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    all_logits = []
    with torch.no_grad():
        for imgs, _ in tqdm(dl, desc=desc):
            imgs = imgs.to(device)
            logits = teacher(imgs)
            all_logits.append(logits.cpu().numpy())
    return np.concatenate(all_logits, axis=0)


# ── Extract & cache ────────────────────────────────────────────────────────
if os.path.exists(LOGITS_CACHE):
    cache_data      = np.load(LOGITS_CACHE, allow_pickle=True).item()
    train_logits    = cache_data['train']
    val_logits      = cache_data['val']
    test_logits     = cache_data['test']
    print(f'Loaded cached logits. Train: {train_logits.shape}')
else:
    train_logits = extract_logits(train_samples, teacher, teacher_tf, torch_device, desc='Train logits')
    val_logits   = extract_logits(val_samples,   teacher, teacher_tf, torch_device, desc='Val logits')
    test_logits  = extract_logits(test_samples,  teacher, teacher_tf, torch_device, desc='Test logits')
    np.save(LOGITS_CACHE, {'train': train_logits, 'val': val_logits, 'test': test_logits})
    print(f'Logits extracted & saved. Train: {train_logits.shape}')

Train logits:   0%|          | 0/407 [00:00<?, ?it/s]

## 4 — TF Data Pipelines (Grayscale 96×96)

In [ ]:
def load_grayscale(path, label):
    """Read image → grayscale 96×96 → float32 in [0,1]."""
    raw = tf.io.read_file(path)
    img = tf.image.decode_jpeg(raw, channels=3)          # decode as RGB
    img = tf.image.rgb_to_grayscale(img)                 # (H,W,1)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])     # (96,96,1)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

# ── Helper: build tf.data from sample list ─────────────────────────────────
def make_dataset(samples, logits=None, augment_fn=None, shuffle=False):
    paths  = [s[0] for s in samples]
    labels = [s[1] for s in samples]

    if logits is not None:
        # Dataset carries (image, label, teacher_logits)
        ds = tf.data.Dataset.from_tensor_slices(
            (paths,
             tf.cast(labels, tf.int32),
             tf.cast(logits, tf.float32))
        )
        ds = ds.map(
            lambda p, l, t: (*load_grayscale(p, l), t),
            num_parallel_calls=tf.data.AUTOTUNE
        )
        if augment_fn:
            ds = ds.map(
                lambda img, lbl, t: (*augment_fn(img, lbl), t),
                num_parallel_calls=tf.data.AUTOTUNE
            )
    else:
        # Normal dataset: (image, label)
        ds = tf.data.Dataset.from_tensor_slices(
            (paths, tf.cast(labels, tf.int32))
        )
        ds = ds.map(load_grayscale, num_parallel_calls=tf.data.AUTOTUNE)
        if augment_fn:
            ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=len(samples), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

In [ ]:
# ── Normal CE pipelines (no teacher logits) ────────────────────────────────
ce_train_ds = make_dataset(train_samples, augment_fn=augment, shuffle=True)
ce_val_ds   = make_dataset(val_samples)
ce_test_ds  = make_dataset(test_samples)

# ── KD pipelines (with teacher logits) ────────────────────────────────────
kd_train_ds = make_dataset(train_samples, logits=train_logits, augment_fn=augment, shuffle=True)
kd_val_ds   = make_dataset(val_samples,   logits=val_logits)
kd_test_ds  = make_dataset(test_samples,  logits=test_logits)

# Quick sanity check
for batch in ce_train_ds.take(1):
    imgs, lbls = batch
    print(f'CE batch — images: {imgs.shape}, labels: {lbls.shape}')

for batch in kd_train_ds.take(1):
    imgs, lbls, tlogits = batch
    print(f'KD batch — images: {imgs.shape}, labels: {lbls.shape}, teacher logits: {tlogits.shape}')

## 5 — Model: TinyAgCNN

In [ ]:
def build_TinyAgCNN(input_shape=(96, 96, 1), num_classes=4):
    """
    3× [Conv3×3 VALID → BN → ReLU → MaxPool2] → Flatten → Dense(128) → Dense(4)
    Input: (96,96,1) grayscale float32
    Feature map sizes: 96→47→22→10 after each Conv+Pool
    Flatten: 10×10×64 = 6400
    """
    inputs = tf.keras.Input(shape=input_shape, name='grayscale_input')

    # Block 1 — 96→47
    x = tf.keras.layers.Conv2D(16, 3, strides=1, padding='valid', use_bias=False)(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPooling2D(2, strides=2)(x)

    # Block 2 — 47→22
    x = tf.keras.layers.Conv2D(32, 3, strides=1, padding='valid', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPooling2D(2, strides=2)(x)

    # Block 3 — 22→10
    x = tf.keras.layers.Conv2D(64, 3, strides=1, padding='valid', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPooling2D(2, strides=2)(x)

    # Classifier
    x = tf.keras.layers.Flatten()(x)                          # 6400
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    outputs = tf.keras.layers.Dense(num_classes, name='logits')(x)  # raw logits

    return tf.keras.Model(inputs, outputs, name='TinyAgCNN')

# Verify shapes once
tmp = build_TinyAgCNN()
tmp.summary()
print(f'\nParam count: {tmp.count_params():,}')

## 6 — Baseline: Normal Cross-Entropy Training

In [ ]:
def run_epoch(model, dataset, optimizer=None, training=True):
    """One CE epoch. Returns (loss, recall_macro, all_preds, all_labels)."""
    running_loss = 0.0
    all_preds, all_labels = [], []
    n_batches = 0

    for imgs, labels in dataset:
        if training:
            with tf.GradientTape() as tape:
                logits = model(imgs, training=True)
                loss   = tf.reduce_mean(
                    tf.keras.losses.sparse_categorical_crossentropy(
                        labels, logits, from_logits=True))
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
        else:
            logits = model(imgs, training=False)
            loss   = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    labels, logits, from_logits=True))

        preds = tf.argmax(logits, axis=1).numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        running_loss += loss.numpy()
        n_batches    += 1

    avg_loss = running_loss / n_batches
    recall   = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, recall, all_preds, all_labels


def train_baseline(model, train_ds, val_ds, epochs, lr, save_path='best_baseline.keras'):
    optimizer   = tf.keras.optimizers.AdamW(learning_rate=lr)
    best_recall = 0.0
    history     = {'train_loss': [], 'val_loss': [], 'train_recall': [], 'val_recall': []}

    for epoch in range(1, epochs + 1):
        tr_loss, tr_rec, _, _ = run_epoch(model, train_ds, optimizer, training=True)
        vl_loss, vl_rec, _, _ = run_epoch(model, val_ds,   training=False)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_recall'].append(tr_rec)
        history['val_recall'].append(vl_rec)

        marker = ''
        if vl_rec > best_recall:
            best_recall = vl_rec
            model.save(save_path)
            marker = '  ← saved'

        print(f'[Baseline] Epoch {epoch:02d}/{epochs}  '
              f'TrLoss={tr_loss:.4f} TrRec={tr_rec:.4f}  '
              f'VlLoss={vl_loss:.4f} VlRec={vl_rec:.4f}{marker}')

    return history

In [ ]:
baseline_model = build_TinyAgCNN()
baseline_history = train_baseline(
    baseline_model,
    ce_train_ds,
    ce_val_ds,
    epochs=BASELINE_EPOCHS,
    lr=BASELINE_LR,
    save_path='best_baseline.keras'
)

## 7 — Knowledge Distillation Training

In [ ]:
def kd_loss(student_logits, teacher_logits, labels, alpha, T):
    """
    alpha  : weight for hard CE loss  (0 = soft only, 1 = CE only)
    T      : temperature
    Hard loss  : sparse CE(student_logits, ground_truth)
    Soft loss  : KL(teacher_soft || student_soft) * T²
    """
    # Hard
    HL = tf.reduce_mean(
        tf.keras.losses.sparse_categorical_crossentropy(
            labels, student_logits, from_logits=True))

    # Soft
    s_soft = tf.nn.log_softmax(student_logits / T, axis=1)
    t_soft = tf.nn.softmax(teacher_logits  / T, axis=1)
    SL = tf.reduce_mean(
        tf.reduce_sum(
            t_soft * (tf.math.log(t_soft + 1e-8) - s_soft),
            axis=1)) * (T ** 2)

    return alpha * HL + (1.0 - alpha) * SL

In [ ]:
def run_kd_epoch(model, dataset, optimizer=None, alpha=0.5, T=4.0, training=True):
    """One KD epoch. Dataset yields (img, label, teacher_logits)."""
    running_loss = 0.0
    all_preds, all_labels = [], []
    n_batches = 0

    for imgs, labels, t_logits in dataset:
        if training:
            with tf.GradientTape() as tape:
                s_logits = model(imgs, training=True)
                loss     = kd_loss(s_logits, t_logits, labels, alpha, T)
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
        else:
            s_logits = model(imgs, training=False)
            loss     = kd_loss(s_logits, t_logits, labels, alpha, T)

        preds = tf.argmax(s_logits, axis=1).numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        running_loss += loss.numpy()
        n_batches    += 1

    avg_loss = running_loss / n_batches
    recall   = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, recall, all_preds, all_labels


def train_kd(model, train_ds, val_ds, epochs, lr, alpha, T,
             save_path='best_kd.keras'):
    optimizer   = tf.keras.optimizers.AdamW(learning_rate=lr)
    best_recall = 0.0
    history     = {'train_loss': [], 'val_loss': [], 'train_recall': [], 'val_recall': []}

    for epoch in range(1, epochs + 1):
        tr_loss, tr_rec, _, _ = run_kd_epoch(model, train_ds, optimizer, alpha, T, training=True)
        vl_loss, vl_rec, _, _ = run_kd_epoch(model, val_ds,                alpha, T, training=False)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_recall'].append(tr_rec)
        history['val_recall'].append(vl_rec)

        marker = ''
        if vl_rec > best_recall:
            best_recall = vl_rec
            model.save(save_path)
            marker = '  ← saved'

        print(f'[KD] Epoch {epoch:02d}/{epochs}  '
              f'TrLoss={tr_loss:.4f} TrRec={tr_rec:.4f}  '
              f'VlLoss={vl_loss:.4f} VlRec={vl_rec:.4f}{marker}')

    return history

In [ ]:
kd_model = build_TinyAgCNN()
kd_history = train_kd(
    kd_model,
    kd_train_ds,
    kd_val_ds,
    epochs=KD_EPOCHS,
    lr=KD_LR,
    alpha=KD_ALPHA,
    T=KD_T,
    save_path='best_kd.keras'
)

## 8 — Evaluation & Comparison

In [ ]:
def evaluate_model(model, test_ds, class_names, title='Model'):
    """Runs on CE test_ds (no teacher logits needed)."""
    all_preds, all_labels = [], []

    for batch in test_ds:
        # Handle both (img, lbl) and (img, lbl, logits)
        imgs, labels = batch[0], batch[1]
        logits = model(imgs, training=False)
        preds  = tf.argmax(logits, axis=1).numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    print(f'\n──────── {title} — Test Set ────────')
    print(f'Macro Recall: {recall:.4f}')
    print(classification_report(all_labels, all_preds, target_names=class_names))

    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — {title}')
    plt.tight_layout()
    plt.savefig(f'cm_{title.replace(" ","_")}.png', dpi=150)
    plt.show()

    return recall, all_preds, all_labels

In [ ]:
# Load best checkpoints
best_baseline = tf.keras.models.load_model('best_baseline.keras')
best_kd       = tf.keras.models.load_model('best_kd.keras')

bl_recall, _, _ = evaluate_model(best_baseline, ce_test_ds, CLASS_NAMES, title='Baseline CE')
kd_recall, _, _ = evaluate_model(best_kd,       ce_test_ds, CLASS_NAMES, title='KD')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Loss
ax = axes[0]
ax.plot(baseline_history['val_loss'],  label='Baseline val loss',  color='steelblue')
ax.plot(kd_history['val_loss'],        label='KD val loss',        color='tomato')
ax.plot(baseline_history['train_loss'], label='Baseline train loss', color='steelblue', linestyle='--', alpha=0.5)
ax.plot(kd_history['train_loss'],       label='KD train loss',       color='tomato',    linestyle='--', alpha=0.5)
ax.set_title('Loss')
ax.set_xlabel('Epoch')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Recall
ax = axes[1]
ax.plot(baseline_history['val_recall'],  label='Baseline val recall',  color='steelblue')
ax.plot(kd_history['val_recall'],        label='KD val recall',        color='tomato')
ax.plot(baseline_history['train_recall'], label='Baseline train recall', color='steelblue', linestyle='--', alpha=0.5)
ax.plot(kd_history['train_recall'],       label='KD train recall',       color='tomato',    linestyle='--', alpha=0.5)
ax.set_title('Macro Recall')
ax.set_xlabel('Epoch')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('Baseline CE vs Knowledge Distillation — TinyAgCNN (Grayscale 96×96)', fontsize=11)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

print(f'\nBaseline best val recall: {max(baseline_history["val_recall"]):.4f}')
print(f'KD       best val recall: {max(kd_history["val_recall"]):.4f}')
print(f'\nBaseline test recall: {bl_recall:.4f}')
print(f'KD       test recall: {kd_recall:.4f}')

## 9 — Export Best Model
Pick whichever scored higher. Save as `.h5` for ESP-DL quantization script.

In [ ]:
winner = best_kd if kd_recall >= bl_recall else best_baseline
winner_name = 'KD' if kd_recall >= bl_recall else 'Baseline'
print(f'Winner: {winner_name} (test recall={max(kd_recall, bl_recall):.4f})')

# SavedModel format (for TFLite / ESP-DL conversion)
winner.save('TinyAgCNN_best.keras')

# Also export as TFLite (int8 quantization ready)
converter = tf.lite.TFLiteConverter.from_keras_model(winner)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Representative dataset for full-int8 quantization
def representative_data_gen():
    for imgs, _ in ce_test_ds.take(50):
        for img in imgs:
            yield [tf.expand_dims(img, 0)]

converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
with open('TinyAgCNN_int8.tflite', 'wb') as f:
    f.write(tflite_model)

print(f'TFLite model size: {len(tflite_model)/1024:.1f} KB')
print('Saved: TinyAgCNN_best.keras  |  TinyAgCNN_int8.tflite')